### Run this notebook online

[![Open in Colab](https://img.shields.io/badge/Open_in-Colab-F9AB00?logo=googlecolab&logoColor=F9AB00)](https://colab.research.google.com/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/files/notebooks/CIFAR10%20distilled%20DiT%20CLF.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2Fhosein-fanai%2FContinual-Learning-with-Diffusion-Vision-Transformers%2Fblob%2Fmain%2Ffiles%2Fnotebooks%2FCIFAR10%2520distilled%2520DiT%2520CLF.ipynb)
[![Launch Binder](https://img.shields.io/badge/launch-binder-F5793A?logo=jupyter&logoColor=white)](https://mybinder.org/v2/gh/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/main?urlpath=lab%2Ftree%2Ffiles%2Fnotebooks%2FCIFAR10%20distilled%20DiT%20CLF.ipynb)

- **Google Colab:** open the notebook, select a GPU for training under **Runtime > Change runtime type**, then choose **Run all**.
- **Kaggle:** sign in and import the notebook, enable **Internet**, select a **GPU** accelerator for training, then **Run all**.
- **Binder:** opens a temporary CPU JupyterLab session. Use it to inspect the notebook or run small checks; full training needs more resources.
- **[Studio Lab](https://studiolab.sagemaker.aws/import/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/files/notebooks/CIFAR10%20distilled%20DiT%20CLF.ipynb) (existing accounts only):** start a runtime, copy the notebook to your project, select a Python **3.11–3.13** kernel, and set `RUNTIME = "studiolab"` in the first code cell before **Run all**. For CPU, also set `CUDA = False`.

The **first code cell** finds or downloads the repository and prepares TensorFlow **2.20** / Keras **3.11.2** before project imports. If setup requests a restart, restart the kernel and run all again. For another hosted Jupyter service, set `RUNTIME = "hosted"` (`CUDA = False` for CPU or compatible provider-managed CUDA). Locally, select the project TensorFlow kernel.

Launch links open the published GitHub `main` version; publish this notebook and its setup files together before using them. For a notebook that has not been published, upload its `.ipynb` file to Colab or Kaggle instead. GPU availability depends on the provider. Save checkpoints and results before a temporary session ends.

See the [hosted runtime guide](https://github.com/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/files/notebooks/thesis/README.md#hosted-runtimes) for setup and import details.


In [ ]:
# Shared setup: use the local initializer when available, otherwise download it.
from pathlib import Path
from urllib.request import urlopen


CHECKOUT_NAME = "Continual-Learning-with-Diffusion-Vision-Transformers"
REPOSITORY = f"https://github.com/hosein-fanai/{CHECKOUT_NAME}.git"
REVISION = "main"
RUNTIME = "auto"  # Use "hosted" for another online service, or "local" to verify only.
CUDA = None  # False: CPU or managed CUDA; True: retain CUDA pip dependencies.

_locations = (Path.cwd(), *Path.cwd().parents, Path.cwd() / CHECKOUT_NAME, 
              Path("/kaggle/working") / CHECKOUT_NAME, Path("/content") / CHECKOUT_NAME)
_initializer = next((path / "files" / "notebooks" / "init.py" for path in _locations
                     if (path / "files" / "notebooks" / "init.py").is_file()), None)
_url = f"https://raw.githubusercontent.com/hosein-fanai/{CHECKOUT_NAME}/{REVISION}/files/notebooks/init.py"
_setup = {"__name__": "notebook_setup", "__file__": str(_initializer or _url)}
with (_initializer.open("rb") if _initializer else urlopen(_url, timeout=30)) as _file:
    exec(compile(_file.read(), _setup["__file__"], "exec"), _setup)
ROOT, RUNTIME_PACKAGES = _setup["prepare_notebook"](
    checkout_name=CHECKOUT_NAME, 
    repository=REPOSITORY, 
    revision=REVISION, 
    runtime=RUNTIME, 
    cuda=CUDA
)

In [ ]:
from common import utils


utils.init()

In [ ]:
from common.dataloader import get_dataset

In [ ]:
epochs = 50
val_freq = 5

In [ ]:
from common.dataloader import load_cifar10


x_train, y_train, _, _, x_test, y_test = load_cifar10(
    indices=None, validation_ratio=0.0, verbose=0
)

trainset = get_dataset(x_train, y_train.reshape(-1))
valset = get_dataset(x_test, y_test.reshape(-1), shuffle_buffer=0, drop_remainder=False)

In [ ]:
from tensorflow.keras import optimizers

from common.model import get_model

from diffusion.models.transformer.di_t_classifier import DiTClassifier
from diffusion.models.wrapper.diffusion_classifier import DiffusionClassifier


model = DiffusionClassifier(
    network=DiTClassifier(
        patchify_with_cnn=True, 
        image_size=32, 
        channels=3, 
        depth=8, 
        dim=128, 
        clf_depth=8
    ), 
    preprocess_type="standardize", 
    use_ema=False, 
    teacher_network=get_model(
        10, 
        model_type="pretrained", 
        conv_base_name="EfficientNetV2L", 
        num_last_not_frozen=20, 
        dropout_rate=0.50
    ), 
    trainable_teacher=True, 
    clf_distil_loss_coef=8.6e-3, 
    clf_distil_type="soft", 
    clf_train_noisy_input_type="noisy", 
    clf_train_class_input_type="null_class_only", 
    mask_by_nulls=False, 
    mask_by_t_threshold=False
)

model.compile(
    optimizer=optimizers.Adam(
        optimizers.schedules.CosineDecay(
            initial_learning_rate=1e-3, 
            decay_steps=epochs * len(trainset)
        )
    ), 
    loss="mse"
)

model.summary()

In [ ]:
from tensorflow.keras import callbacks


teacher_history = model.fit_teacher(
    trainset, 
    epochs=100, 
    validation_data=valset, 
    callbacks=[
        callbacks.EarlyStopping(
            monitor="val_accuracy", 
            patience=10, 
            restore_best_weights=True, 
            min_delta=0.01, 
            verbose=1
        ), 
        callbacks.ReduceLROnPlateau(
            monitor="val_accuracy", 
            patience=4, 
            min_delta=0.01, 
            factor=0.8, 
            verbose=1
        )
    ]
).history

In [ ]:
from diffusion import ImageGenerator
from common.callbacks.lr_logger import LrLogger


history = model.fit(
    trainset, 
    epochs=epochs, 
    validation_data=valset, 
    validation_freq=val_freq, 
    callbacks=[
        LrLogger(), 
        callbacks.ProgbarLogger(), 
        ImageGenerator(frequency=val_freq)
    ]
).history

In [ ]:
model.save_weights("./models/CIFAR10 distilled DiT CLF/model.weights.h5")

In [ ]:
model.evaluate(valset, network_name="raw")

In [ ]:
model.evaluate_ensemble_accuracy(
    valset, 
    t_chunk_size=8, 
    max_t=256, 
    t_range_drop_rate=0.75
)

In [ ]:
model.evaluate_ensemble_accuracy(
    valset, 
    t_chunk_size=8, 
    max_t=256, 
    t_range_drop_rate=0.99
)

In [ ]:
model.evaluate_ensemble_accuracy(
    valset, 
    t_chunk_size=8, 
    max_t=128, 
    t_range_drop_rate=0.75
)

In [ ]:
model.evaluate_ensemble_accuracy(
    valset, 
    t_chunk_size=8, 
    max_t=128, 
    t_range_drop_rate=0.99
)

In [ ]:
model.evaluate_ensemble_accuracy(
    valset, 
    t_chunk_size=8, 
    max_t=64, 
    t_range_drop_rate=0.75
)

In [ ]:
model.evaluate_ensemble_accuracy(
    valset, 
    t_chunk_size=8, 
    max_t=64, 
    t_range_drop_rate=0.99
)

In [ ]:
model.evaluate_ensemble_accuracy(
    valset, 
    t_chunk_size=8, 
    max_t=32, 
    t_range_drop_rate=0.75
)

In [ ]:
model.evaluate_ensemble_accuracy(
    valset, 
    t_chunk_size=8, 
    max_t=32, 
    t_range_drop_rate=0.99
)

In [ ]:
model.evaluate_ensemble_accuracy(
    valset, 
    t_chunk_size=8, 
    max_t=16, 
    t_range_drop_rate=0.75
)

In [ ]:
model.evaluate_ensemble_accuracy(
    valset, 
    t_chunk_size=8, 
    max_t=16, 
    t_range_drop_rate=0.99
)

In [ ]:
model.evaluate_ensemble_accuracy(
    valset, 
    t_chunk_size=2, 
    max_t=8
)

In [ ]:
model.evaluate_ensemble_accuracy(
    valset, 
    t_chunk_size=2, 
    max_t=4
)

In [ ]:
# model.evaluate_ensemble_accuracy(
#     valset, 
#     separate_probas=True, 
#     t_chunk_size=1, 
#     max_t=64, 
#     t_range_drop_rate=0.75
# )

In [ ]:
from common.utils import plot_images


imgs = model.sample(add_null_label=True, scale=3., eta=1., steps=1_000, verbose=True)
plot_images(imgs, has_null_label=True)

In [ ]:
imgs = model.sample(add_null_label=True, scale=3., steps=1_000, verbose=True)
plot_images(imgs, has_null_label=True)

In [ ]:
imgs = model.sample(add_null_label=True, scale=3., verbose=True)
plot_images(imgs, has_null_label=True)

In [ ]:
imgs = model.sample(add_null_label=True, scale=4., verbose=True)
plot_images(imgs, has_null_label=True)

In [ ]:
from common.utils import plot_history


plot_history(history, show_all_x_ticks=False, validation_freq=val_freq)

In [ ]:
plot_history(history, show_all_x_ticks=False, range_=(9, None), validation_freq=val_freq)